# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainDev04/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

**Lane 3 — Structured Content Archetype Clustering** (provisional, carried from ML-02).

ML-02 asked whether this question is worth seven weeks. ML-03 asks what machine-learning shape it has —
and for an unsupervised lane that means answering a question the supervised lanes never have to:
**how do you define success when there is nothing to be right about?**

> Skills for this card: `skills/framing-ml-problems/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.
> Data: `data/raw/content_refresh_anonymized.csv`. Every number computed live; every threshold labelled
> as a choice.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

SEED = 42
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ZainDev04/Flyrank-ML-Internship"   # <-- change to YOUR fork
CSV_REL  = "data/raw/content_refresh_anonymized.csv"
if IN_COLAB and not Path(CSV_REL).exists():
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    here = Path.cwd()
    for c in [here, *here.parents]:
        if (c / CSV_REL).exists():
            os.chdir(c); break
assert Path(CSV_REL).exists()
pd.set_option("display.width", 190); pd.set_option("display.max_columns", 40)

MIN_IMPRESSIONS, K = 300, 6           # both carried from ML-02, both policy choices
df = pd.read_csv(CSV_REL)
d = df[(df["impressions_90d"] >= MIN_IMPRESSIONS) & (df["avg_position"] > 0)].copy()
d["log_impressions"] = np.log1p(d["impressions_90d"])
d["log_clicks"] = np.log1p(d["clicks_90d"])
d["log_sessions"] = np.log1p(d["sessions_90d"])
d["impression_consistency"] = d["days_with_impressions"] / 90
d = d.reset_index(drop=True)

FEATURES = ["log_impressions", "log_clicks", "ctr", "avg_position", "log_sessions",
            "engagement_rate", "impression_consistency", "content_age_days",
            "days_since_last_update"]
X = StandardScaler().fit_transform(d[FEATURES].fillna(d[FEATURES].median()))
d["cluster"] = KMeans(n_clusters=K, n_init=25, random_state=SEED).fit_predict(X)

print(f"eligible inventory : {len(d):,} pages / {d['client_id'].nunique()} clients")
print(f"features           : {len(FEATURES)}")
print(f"k                  : {K}   (silhouette peak from ML-02)")
print(f"cluster sizes      : {sorted(d['cluster'].value_counts().tolist())}")

eligible inventory : 18,752 pages / 29 clients
features           : 9
k                  : 6   (silhouette peak from ML-02)
cluster sizes      : [537, 985, 3595, 4113, 4598, 4924]


## 1. My lane as an ML task (type)

**Clustering — unsupervised.** Not classification, not ranking, and the reason is that I have no observed
outcome I want to reproduce.

The `framing-ml-problems` task table is explicit: *"What kinds of items exist?" → clustering → no target →
silhouette + human sense-check.* That is my question verbatim. The moment I write down a target, I have
changed lanes.

Why not the alternatives:

| Task type | Why it's wrong for this question |
|---|---|
| **Classification** | Requires a label. The only labels available are `trend_direction`-derived — rules someone else wrote. Predicting them means learning their rule, not finding structure. And a classifier can only ever find groups I already named. |
| **Ranking** | Answers "which first?", not "what kinds?". It also assumes a single axis of badness, which is the assumption I am specifically trying to avoid — a stale page and an intermittent page are not two points on one scale. |
| **Regression** | Needs a continuous target. Same objection as classification. |

**The consequence I have to live with.** Unsupervised methods always return an answer. K-Means will hand
me six clusters from pure noise, and they will have profiles, and I will be able to name them. Nothing in
the algorithm will warn me. That is the central methodological risk of this lane, and sections 3 and 5 are
almost entirely about defending against it.

## 2. Target or proxy

### There is no target. That is the honest answer, and it needs unpacking rather than hiding.

There is no column I am predicting and no ground truth I can be scored against. Nobody has labelled these
pages with their true archetype, and no such labelling exists to be discovered — "archetype" is a construct
I am imposing to make policy tractable, not a hidden property of the page.

So the thing that stands in for a target is the **cluster assignment itself**, and the thing that stands in
for correctness is whether that assignment is *stable, externally coherent, and actionable*. Section 3
turns each of those into a number.

### What I am NOT doing, stated so I cannot drift into it

- **Not predicting cluster membership.** Fitting a classifier to the cluster labels would score ~perfectly
  and mean nothing — the labels came from the features. That is circular, and it is the unsupervised
  version of the leakage trap the supervised lanes fight.
- **Not treating clusters as ground truth for anything downstream.** If a later notebook uses
  `cluster` as a feature or a label, the reasoning becomes self-confirming.
- **Not clustering on the product's own answers.** `health_score`, `priority_score`, `action_type` are not
  in this dataset by design. Clustering on them would rediscover FlyRank's rules and call it a finding.

### The proxy I *do* commit to: the action mapping

Each cluster gets exactly one recommended action from the lane guide's vocabulary — protect, improve,
rewrite, merge, prune, monitor. That mapping is the closest thing this project has to a testable claim,
because it is falsifiable in a way a cluster is not: *if two clusters would receive the same action, they
did not need to be separate clusters.* That is a real constraint, and I check it in ML-08 rather than
assuming it.

## 3. Success metric

There is no accuracy here, so "good" has to be built out of several weaker signals that fail in different
ways. I commit to four, in priority order, and to the numbers that count as passing — before seeing them.

### Metric 1 — Stability. Do the clusters survive being re-run?
The cheapest way to be fooled is to accept whatever the first random seed produced. Adjusted Rand Index
compares two partitions: 1.0 identical, 0.0 no better than chance.

**Bar: mean ARI ≥ 0.80 across seeds and across 80% subsamples.**

In [2]:
base = d["cluster"].to_numpy()
seed_ari = [adjusted_rand_score(base, KMeans(n_clusters=K, n_init=25, random_state=s).fit_predict(X))
            for s in (1, 7, 13, 99, 2024)]
sub_ari = []
for s in (1, 7, 13, 99, 2024):
    idx = np.random.RandomState(s).choice(len(X), int(0.8 * len(X)), replace=False)
    lab = KMeans(n_clusters=K, n_init=25, random_state=SEED).fit_predict(X[idx])
    sub_ari.append(adjusted_rand_score(base[idx], lab))

print(f"ARI across 5 random seeds      : {np.round(seed_ari, 3).tolist()}  mean {np.mean(seed_ari):.3f}")
print(f"ARI across 5 x 80% subsamples  : {np.round(sub_ari, 3).tolist()}  mean {np.mean(sub_ari):.3f}")
print(f"\nBar was 0.80. Both pass comfortably.")
print("Meaning: the partition is a property of the data and the feature set, not of where")
print("the algorithm happened to start, and not of which 80% of pages I happened to have.")

ARI across 5 random seeds      : [0.999, 1.0, 1.0, 0.995, 0.997]  mean 0.998
ARI across 5 x 80% subsamples  : [0.995, 0.993, 0.985, 0.989, 0.992]  mean 0.991

Bar was 0.80. Both pass comfortably.
Meaning: the partition is a property of the data and the feature set, not of where
the algorithm happened to start, and not of which 80% of pages I happened to have.


### Metric 2 — Separation. How well-defined are the boundaries?

**Bar: silhouette > 0.15, reported honestly whatever it is.** I am deliberately setting a low bar and
saying why: behavioural metrics form continua, not islands. A silhouette of 0.6 on this data would make me
suspect I had accidentally clustered on one dominant variable.

In [3]:
sample = np.random.RandomState(0).choice(len(X), 5000, replace=False)
sil = silhouette_score(X[sample], d["cluster"].to_numpy()[sample])
print(f"silhouette at k={K}: {sil:.3f}")
print()
print("Passes the 0.15 bar - and it is WEAK, which I report rather than bury.")
print("These are regions of a continuum with soft edges, not separated islands. Every")
print("downstream sentence has to be compatible with that, which is why the deliverable")
print("is 'a defensible way to route policy' and never 'the natural kinds of content'.")

silhouette at k=6: 0.221

Passes the 0.15 bar - and it is WEAK, which I report rather than bury.
These are regions of a continuum with soft edges, not separated islands. Every
downstream sentence has to be compatible with that, which is why the deliverable
is 'a defensible way to route policy' and never 'the natural kinds of content'.


### Metric 3 — External coherence. Do the clusters differ on things the algorithm never saw?

This is the metric I trust most, and it is the one silhouette cannot give me. Silhouette only asks whether
the clusters are tight *in the space I built them in* — it is graded on its own homework. External
coherence asks something harder: **do these groups also differ on variables that were not in the feature
set at all?**

If they do, the structure is a property of the pages rather than of my nine columns.

**Bar: at least three held-out variables separate across clusters at p < 0.001, with medians that differ
by a practically meaningful amount — not just a statistically significant one on 18,752 rows.**

In [4]:
HELD_OUT = ["word_count", "char_count", "search_volume", "competition", "cpc"]
assert not set(HELD_OUT) & set(FEATURES), "a held-out variable is in the feature set - not held out"

print("Median value per cluster - NONE of these columns were used to build the clusters:\n")
print(d.groupby("cluster")[HELD_OUT].median().round(1).to_string())
print("\nThe keyword columns are heavily zero-inflated, so a median hides the difference.")
print("For those, share-with-data and the mean say more:\n")
for c in ["search_volume", "competition", "cpc"]:
    t = d.groupby("cluster")[c].agg(pct_with_data=lambda s: (s.fillna(0) > 0).mean() * 100,
                                    mean="mean", p75=lambda s: s.quantile(0.75))
    print(f"  {c}:")
    print("    " + t.round(2).to_string().replace("\n", "\n    "))

print("\nKruskal-Wallis (non-parametric, uses the whole distribution, not the median):")
for c in HELD_OUT:
    g = [v[c].dropna() for _, v in d.groupby("cluster")]
    h, p = stats.kruskal(*g)
    med = d.groupby("cluster")[c].median()
    mean = d.groupby("cluster")[c].mean()
    if med.min() > 0:
        spread = f"median {med.min():.0f}-{med.max():.0f} ({med.max()/med.min():.1f}x)"
    else:
        spread = f"median near 0; mean {mean.min():.2f}-{mean.max():.2f} ({mean.max()/max(mean.min(),1e-9):.1f}x)"
    print(f"  {c:15s} H={h:8.1f}  p={p:.2e}  {spread}")

Median value per cluster - NONE of these columns were used to build the clusters:

         word_count  char_count  search_volume  competition  cpc
cluster                                                         
0            2789.0     17634.0           20.0          0.0  0.0
1            4428.0     29490.5           10.0          0.0  0.0
2            3174.5     20976.5           10.0          0.0  0.0
3            2772.0     18732.0           10.0          0.0  0.0
4            2871.0     19082.0            0.0          0.0  0.0
5            2733.0     17968.0           10.0          0.0  0.0

The keyword columns are heavily zero-inflated, so a median hides the difference.
For those, share-with-data and the mean say more:

  search_volume:
             pct_with_data    mean    p75
    cluster                              
    0                88.29  423.36  110.0
    1                52.61  203.23   20.0
    2                53.37   62.96   20.0
    3                63.96  202.84   

In [5]:
print("And two held-out CATEGORICAL variables, where the answer is different:\n")
print("content_type share per cluster:")
print(pd.crosstab(d["cluster"], d["content_type"], normalize="index").round(3).to_string())
print("\nmain_intent share per cluster:")
print(pd.crosstab(d["cluster"], d["main_intent"], normalize="index").round(3).to_string())
print()
print("These are nearly FLAT across clusters - roughly 57-69% informational everywhere,")
print("99% keyword article everywhere. That is a good result, not a failed one: it means")
print("my clusters are behavioural groupings, NOT content_type in disguise. If cluster 3")
print("had been 90% feedly article I would be looking at a metadata artefact.")

And two held-out CATEGORICAL variables, where the answer is different:

content_type share per cluster:
content_type  comparison article  feedly article  keyword article
cluster                                                          
0                          0.000           0.004            0.996
1                          0.000           0.013            0.987
2                          0.000           0.005            0.994
3                          0.011           0.057            0.932
4                          0.030           0.008            0.962
5                          0.002           0.009            0.989

main_intent share per cluster:
main_intent  commercial  informational  navigational  transactional
cluster                                                            
0                 0.170          0.627         0.001          0.201
1                 0.186          0.564         0.000          0.250
2                 0.180          0.590         0.001          0.

**Read, and read carefully — the first version of this cell nearly fooled me.** Every numeric held-out
variable separates at p < 1e-100. But p-values on 18,752 rows are cheap, so the question is whether the
differences are *practically* meaningful, and the answer differs by column:

- **Content length is clearly separated on medians.** `word_count` runs 2,733 → 4,428 (1.6×) and
  `char_count` 17,634 → 29,490 (1.7×) across clusters. That is a real difference in the kind of page.
- **The keyword-economics columns have medians at or near zero**, so a median comparison of them is
  meaningless — my first draft printed a "20,000,000,000×" spread, which is what dividing by zero looks
  like when you are not paying attention. Summarised properly, they separate hard anyway: the share of
  pages *with any keyword data* runs **45% → 88%** across clusters, and mean search volume **44 → 423**.

So the bar I set — three held-out variables, p < 0.001, meaningful spread — is met by `word_count`,
`char_count` and `search_volume`, with `competition` and `cpc` supporting. The structure is a property of
the pages, not of my nine columns.

Meanwhile `content_type` and `main_intent` are almost flat. **Both results are needed.** The first says
the groups are real; the second says they are not a restatement of a metadata column.

### Metric 4 — Human sense-check and actionability

The one that cannot be automated, and the one the skill insists on: *name clusters after inspecting them,
never before.* The bar is a sentence, not a number —

> **Can I describe each cluster in one sentence a content lead would recognise, and attach an action they
> would actually take? And do any two clusters get the same action?**

If two clusters map to the same action, they did not need to be separate for *this decision*, and the
honest move is to say so rather than invent a distinction. That check runs in ML-08.

### What I am explicitly NOT using as a metric
- **Inertia / within-cluster sum of squares alone.** It falls monotonically with k. Optimising it picks
  k = n.
- **Any accuracy against the hand-written archetypes from ML-02.** Agreeing with my own rule is not
  validation — it is my rule reflected back. It gets reported in ML-08 as a *comparison*, never as a score.

## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (a page), for one pseudonymized client, over one trailing
90-day window.** Not a client, not a day, not a keyword, not a cluster. The page is what gets routed to a
treatment, so the page is the unit.

In [6]:
assert d["content_id"].is_unique, "grain broken: content_id must identify a row uniquely"
print(f"Grain check passed: {d['content_id'].nunique():,} unique ids in {len(d):,} rows.\n")

UNIT = ["content_id", "client_id",
        "log_impressions", "log_clicks", "ctr", "avg_position", "log_sessions",
        "engagement_rate", "impression_consistency", "content_age_days",
        "days_since_last_update", "cluster"]
d[UNIT].head(8).reset_index(drop=True)

Grain check passed: 18,752 unique ids in 18,752 rows.



,content_id,client_id,log_impressions,log_clicks,ctr,avg_position,log_sessions,engagement_rate,impression_consistency,content_age_days,days_since_last_update,cluster
0,content_304f48230142,client_f369cb89fc,8.243808,3.401197,0.76,10.6,2.890372,5.88,0.977778,187,20,2
1,content_a1fb4e703a9e,client_4e07408562,9.636980,2.079442,0.05,20.3,2.302585,0.00,0.977778,445,25,0
2,content_9aa793d4d895,client_7f2253d7e2,9.440023,2.484907,0.09,36.5,2.484907,0.00,0.977778,141,20,4
3,content_331d6c4de07b,client_19581e27de,9.371779,4.077537,0.49,6.2,4.369448,1.28,0.977778,463,22,2
4,content_d99b7a2d90ca,client_3fdba35f04,9.859588,3.218876,0.13,44.0,4.983607,0.00,0.977778,263,14,2
5,content_d4084a4bc775,client_f369cb89fc,8.286773,0.693147,0.03,8.5,1.791759,0.00,0.977778,147,20,4
6,content_a63219c6e95a,client_19581e27de,7.452982,0.693147,0.06,21.2,3.367296,3.57,0.977778,445,22,0
7,content_5e6c160719bc,client_6208ef0f77,10.391300,3.401197,0.09,46.0,4.234107,5.88,0.922222,90,20,2


In [7]:
roles = pd.DataFrame([
 ("content_id",             "context",  "pseudonym; row key; never a feature"),
 ("client_id",              "context",  "pseudonym; used to CHECK clusters aren't clients; never a feature"),
 ("log_impressions",        "feature",  "exposure; log because traffic is heavy-tailed"),
 ("log_clicks",             "feature",  "demand captured; log for the same reason"),
 ("ctr",                    "feature",  "conversion of exposure; x100 percentage (0.36 = 0.36%)"),
 ("avg_position",           "feature",  "where it ranks; rows with 0 ('no data') already excluded"),
 ("log_sessions",           "feature",  "on-site arrivals"),
 ("engagement_rate",        "feature",  "on-site behaviour - SEE CAVEAT BELOW, 59% are zero"),
 ("impression_consistency", "feature",  "days_with_impressions / 90; steady vs spiky exposure"),
 ("content_age_days",       "feature",  "how long it has had to establish itself"),
 ("days_since_last_update", "feature",  "editorial recency"),
 ("word_count",             "held out", "VALIDATION only - external coherence check"),
 ("char_count",             "held out", "VALIDATION only"),
 ("search_volume",          "held out", "VALIDATION only"),
 ("competition",            "held out", "VALIDATION only"),
 ("cpc",                    "held out", "VALIDATION only"),
 ("content_type",           "held out", "VALIDATION only - is the cluster a metadata artefact?"),
 ("main_intent",            "held out", "VALIDATION only"),
 ("trend_pct",              "excluded", "shipped pipeline's label source; imposing it changes the question"),
 ("trend_direction",        "excluded", "same"),
 ("provider_used",          "excluded", "generation metadata, not observed behaviour"),
 ("model_used",             "excluded", "generation metadata, not observed behaviour"),
], columns=["column", "role", "why"])
for r in ["context", "feature", "held out", "excluded"]:
    print(f"  {r:10s} {len(roles[roles.role == r]):2d} columns")
assert set(roles.loc[roles.role == "feature", "column"]) == set(FEATURES), "role table and FEATURES disagree"
print("\nGuard passed: the role table and the actual feature set match exactly.\n")
roles

  context     2 columns
  feature     9 columns
  held out    7 columns
  excluded    4 columns

Guard passed: the role table and the actual feature set match exactly.



,column,role,why
0,content_id,context,pseudonym; row key; never a feature
1,client_id,context,pseudonym; used to CHECK clusters aren't clients; never a feature
2,log_impressions,feature,exposure; log because traffic is heavy-tailed
3,log_clicks,feature,demand captured; log for the same reason
4,ctr,feature,conversion of exposure; x100 percentage (0.36 = 0.36%)
5,avg_position,feature,where it ranks; rows with 0 ('no data') already excluded
6,log_sessions,feature,on-site arrivals
7,engagement_rate,feature,"on-site behaviour - SEE CAVEAT BELOW, 59% are zero"
8,impression_consistency,feature,days_with_impressions / 90; steady vs spiky exposure
9,content_age_days,feature,how long it has had to establish itself


**The caveat that rides with `engagement_rate`.** ML-02 measured it: 59% of eligible pages record
`engagement_rate = 0`, and the share with any engagement climbs 21% → 62% → 90% with impression volume. So
this feature partly encodes volume. I am keeping it because it isolates a genuinely distinct minority, but
**no cluster separated by this feature may be called "disengaged content"** — the honest name has to
acknowledge that low measured engagement and low measurement are indistinguishable here. ML-08 names that
cluster accordingly.

## 5. Why ML beats a fixed rule here

The honest starting point is that a fixed rule is *not bad*. ML-02 built one — a five-rung ladder — and it
produced sensible piles. So the case has to be specific.

In [8]:
def rule_archetype(r):
    if r["impressions_90d"] >= 3000 and r["avg_position"] <= 10:          return "CHAMPION"
    if r["days_since_last_update"] >= 90 and r["impressions_90d"] >= 500: return "STALE_VISIBLE"
    if r["engagement_rate"] >= 10:                                        return "ENGAGED_NICHE"
    if r["days_with_impressions"] < 45:                                   return "INTERMITTENT"
    return "STEADY_LOW"

d["rule_archetype"] = d.apply(rule_archetype, axis=1)
print(f"Agreement between the learned clusters and the hand-written ladder:")
print(f"  ARI = {adjusted_rand_score(d['cluster'], d['rule_archetype']):.3f}\n")
print("Where they land (rows = learned cluster, normalised across the row):")
print(pd.crosstab(d["cluster"], d["rule_archetype"], normalize="index").round(2).to_string())
print("\nCluster sizes:", d["cluster"].value_counts().sort_index().tolist())

Agreement between the learned clusters and the hand-written ladder:
  ARI = 0.288

Where they land (rows = learned cluster, normalised across the row):
rule_archetype  CHAMPION  ENGAGED_NICHE  INTERMITTENT  STALE_VISIBLE  STEADY_LOW
cluster                                                                         
0                   0.13           0.05          0.00           0.01        0.81
1                   0.12           0.00          0.00           0.76        0.11
2                   0.60           0.01          0.00           0.24        0.15
3                   0.03           0.05          0.26           0.06        0.60
4                   0.13           0.05          0.00           0.00        0.82
5                   0.08           0.69          0.00           0.23        0.00

Cluster sizes: [3595, 4598, 4924, 985, 4113, 537]


In [9]:
# The specific thing the rule cannot do: see inside its own biggest bucket.
steady = d[d["rule_archetype"] == "STEADY_LOW"]
print(f"The ladder's biggest bucket, STEADY_LOW, holds {len(steady):,} pages "
      f"({len(steady)/len(d)*100:.0f}% of the inventory).")
print("The learned clusters split it like this:\n")
split = steady.groupby("cluster").agg(
    pages=("content_id", "size"), impressions=("impressions_90d", "median"),
    ctr=("ctr", "median"), position=("avg_position", "median"),
    age_days=("content_age_days", "median"), since_update=("days_since_last_update", "median"),
    consistency=("impression_consistency", "median"))
print(split[split["pages"] > 200].round(2).to_string())
print("\nTwo of those groups are large and differ on AGE by a factor of ~4 at nearly identical")
print("volume and position. Under the ladder they get one label and one treatment.")

The ladder's biggest bucket, STEADY_LOW, holds 8,130 pages (43% of the inventory).
The learned clusters split it like this:

         pages  impressions   ctr  position  age_days  since_update  consistency
cluster                                                                         
0         2920       1200.0  0.09      17.8     445.0          22.0         0.98
1          522        399.0  0.00      17.4     284.0         104.0         0.87
2          735      10620.0  0.42      14.4     147.0          20.0         0.98
3          588        430.0  0.00      12.8     144.0          20.0         0.66
4         3365       1339.0  0.15      13.4     123.0          20.0         0.94

Two of those groups are large and differ on AGE by a factor of ~4 at nearly identical
volume and position. Under the ladder they get one label and one treatment.


### The claim, stated narrowly enough to defend

Not "clustering beats rules." The defensible version:

> A hand-written ladder assigns a page to the first bucket whose threshold it crosses, so **its buckets can
> only be as good as the edges I thought of in advance**. Agreement with the learned partition is
> **ARI ≈ 0.29** — the two overlap substantially where the ladder has a sharp idea (its stale and engaged
> rungs map cleanly onto single learned clusters) and diverge exactly where the ladder is vague. Its
> catch-all bucket, **43% of the whole inventory**, splits into groups that differ ~4× on content age at
> nearly identical volume and position. I did not know to put age in the ladder, which is the point: the
> clustering found a distinction I had not thought to encode.

**And the honest counter-argument, which I will keep testing.** Now that I *know* age matters, I could add
a rung for it — and then the ladder would find that split too. This is a real objection. My answer is that
it holds for one iteration: the value of the unsupervised pass is that it surfaces distinctions to encode,
and it will keep doing that on the warehouse where I have far more columns and no intuition about most of
them. If, by ML-08, everything the clusters find turns out to be re-expressible as two or three rules a
human would write, **I will say so** — and that result would push me toward Lane 1.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — pseudonymized ids only
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### The success bar, fixed before results
| Metric | Bar | Why this one |
|---|---|---|
| Stability (ARI, seeds + subsamples) | ≥ 0.80 | catches "the algorithm made this up" |
| Silhouette | > 0.15, reported honestly | catches "there is no structure at all" |
| External coherence | ≥ 3 held-out variables at p < 0.001 with meaningful spread | catches "the structure is only in my feature set" |
| Human sense-check | one sentence per cluster, one action, no duplicate actions | catches "technically valid, useless" |

### What I owe ML-04 (the data contract)
1. **Rebuild this on the warehouse daily table** — a 90-day aggregate hides whether a page is *becoming* an
   archetype or has been one for a year, and archetype work without trajectory is a photograph of a film.
2. **Handle the `engagement_rate` zero-inflation properly** — either a two-part feature (has-engagement
   flag + rate-when-present) or drop it and see whether the cluster survives.
3. **Missingness audit** — `word_count` is missing along `content_type` lines; it is held out today, but if
   it ever becomes a feature it needs a `has_` flag, never a `fillna(0)`.
4. **Check the client-concentration weakness** — one cluster is 54% a single client, which is the number
   most likely to be hiding a house style rather than an archetype.